# P0-C: Real External vs Parametric Memory

This notebook runs the resumable P0-C experiment. Start with `smoke`; only move to `pilot` after the manifest is fully verified. Results are saved to Google Drive after every lesson/seed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/donleaveher/plasticity-placement.git'
BRANCH = 'agent/add-lora-evaluation'
REPO_DIR = Path('/content/plasticity-placement')
DRIVE_ROOT = Path('/content/drive/MyDrive/plasticity-p0c')
CODE_REVISION_FILE = DRIVE_ROOT / 'code-revision-v2.txt'
SMOKE_DIR = DRIVE_ROOT / 'smoke-v2'
CALIBRATION_DIR = DRIVE_ROOT / 'calibration-v2'
PILOT_DIR = DRIVE_ROOT / 'pilot-v2'
for path in (SMOKE_DIR, CALIBRATION_DIR, PILOT_DIR):
    path.mkdir(parents=True, exist_ok=True)
print('Smoke results:', SMOKE_DIR)

In [ ]:
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['pip', 'install', '-q', 'uv'], check=True)
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
if CODE_REVISION_FILE.exists():
    code_revision = CODE_REVISION_FILE.read_text().strip()
else:
    code_revision = subprocess.run(
        ['git', '-C', str(REPO_DIR), 'rev-parse', f'origin/{BRANCH}'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    revision_tmp = CODE_REVISION_FILE.with_suffix('.tmp')
    revision_tmp.write_text(code_revision + '\n')
    revision_tmp.replace(CODE_REVISION_FILE)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', code_revision], check=True)
print('Pinned code revision:', code_revision)
subprocess.run(['uv', 'sync', '--extra', 'train', '--extra', 'colab'], cwd=REPO_DIR, check=True)

In [ ]:
# Pure-data preparation is deterministic and safe to rerun.
subprocess.run([
    'uv', 'run', 'plasticity-p0c', 'prepare',
    '--output', str(SMOKE_DIR),
], cwd=REPO_DIR, check=True)

In [ ]:
# The manifest resumes verified work after a Colab disconnect.
command = [
    'uv', 'run', 'plasticity-p0c', 'run',
    '--output', str(SMOKE_DIR),
    '--tier', 'smoke',
    '--use-4bit',
    '--rank', '8',
    '--alpha', '16',
    '--learning-rate', '2e-4',
    '--max-steps', '16',
]
subprocess.run(command, cwd=REPO_DIR, check=True)

In [ ]:
subprocess.run([
    'uv', 'run', 'plasticity-p0c', 'aggregate',
    '--output', str(SMOKE_DIR),
], cwd=REPO_DIR, check=True)

import json
summary_path = SMOKE_DIR / 'results' / 'aggregate' / 'summary.json'
summary = json.loads(summary_path.read_text())
summary['arm_summary'], summary['contrasts']

## Development calibration

Do not treat smoke output as evidence. Inspect `manifest.json`; every unit must be `verified` and rollback exact match must be 1.0. Calibration trains 60 development adapters and can resume after a disconnect.

In [ ]:
RUN_CALIBRATION = False  # set True only after smoke is verified
if RUN_CALIBRATION:
    subprocess.run([
        'uv', 'run', 'plasticity-p0c', 'calibrate',
        '--output', str(CALIBRATION_DIR),
        '--use-4bit',
    ], cwd=REPO_DIR, check=True)
    report = json.loads((CALIBRATION_DIR / 'calibration_report.json').read_text())
    print('Selected config:', report['selected_config'])

## Eight-lesson pilot

Run only when calibration produced a non-null `selected_config`. The pilot uses a new Drive directory and the frozen development configuration.

In [ ]:
RUN_PILOT = False
calibration_report = CALIBRATION_DIR / 'calibration_report.json'
if RUN_PILOT:
    if not calibration_report.exists():
        raise FileNotFoundError('Run calibration first')
    subprocess.run([
        'uv', 'run', 'plasticity-p0c', 'run',
        '--output', str(PILOT_DIR),
        '--tier', 'pilot',
        '--use-4bit',
        '--calibration-config', str(calibration_report),
    ], cwd=REPO_DIR, check=True)
    subprocess.run([
        'uv', 'run', 'plasticity-p0c', 'aggregate',
        '--output', str(PILOT_DIR),
    ], cwd=REPO_DIR, check=True)